## Управление умной лампой жестами
## Smart lamp gesture control
 
### Цель: Реализовать механизм управления лампой через жесты и распознавание жестов
### Goal: Implement gesture recognition and control for a smart lightbulb

### Суть алгоритма:
MediaPipe расставляет точки на ладони, мы математически сравниваем их, чтобы распознать жест.
- Жесты считываются только когда рука сжата в кулак (расстояние от кончика пальца до запястья минимально)
- Свайп влево = уменьшение яркости
- Свайп вправо = увеличение яркости
- Свайп вверх = Включить
- Свайп вниз = Выключить

### Algorithm core:
MediaPipe outputs hand landmarks and we use them to determine gesture
- Gestures are recognized only when the hand is closed (distance between the fingertip and wrist is minimal)
- Swipe left = Decrease brightness
- Swipe right = Increase brightness
- Swipe up = Turn on
- Swipe down = Turn off

### Первым делом импортируем библиотеки.
 
First things first let's import libraries.

In [ ]:
# Надо установить модель
# Download the MediaPipe Hand Landmarker model
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np
import time
import uuid
import os
import threading

latest_result = None

delta_x, delta_y = 0, 0

HISTORY_LENGTH = 15
SWIPE_THRESHOLD = 0.2

LAMP_KEY= 0 # your lamp key
LAMP_IP = 0# your lamp ip
LAMP_ID = 0# your lamp id

lamp_is_on = True
lamp_lock = threading.Lock()
brightness = 0

### Будем использовать асинхронный режим модели, поэтому нужен калбек для результата

### We'll use the async detection mode, so a callback us required to handle the results

In [ ]:
def receive_result(result, output_image: mp.Image, timestamp_ms: int):
    global latest_result
    if result.hand_landmarks:
        print(f"[{timestamp_ms}ms] Hands detected: {len(result.hand_landmarks)}")
        latest_result = result

### Настраиваем модель:
- Одна рука
- Транслирование видео с вебкамеры
- Настраиваем уверенность

### Configuring the model:
- Single hand
- Live stream from webcam
- Configuring confidence thresholds

In [ ]:
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, 
                                       num_hands=1, 
                                       running_mode=mp.tasks.vision.RunningMode.LIVE_STREAM,
                                       min_hand_detection_confidence=0.3,
                                       min_hand_presence_confidence=0.5,
                                       min_tracking_confidence=0.3,
                                       result_callback=receive_result)

### Функция из документации google, которая отрисовывает точки на руке
### Google documentation function for displaying landmarks

In [ ]:
mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54) # vibrant green

def draw_landmarks_on_image(rgb_image, detection_result):
  hand_landmarks_list = detection_result.hand_landmarks
  handedness_list = detection_result.handedness
  annotated_image = np.copy(rgb_image)

  # Loop through the detected hands to visualize.
  for idx in range(len(hand_landmarks_list)):
    hand_landmarks = hand_landmarks_list[idx]
    handedness = handedness_list[idx]

    # Draw the hand landmarks.
    mp_drawing.draw_landmarks(
      annotated_image,
      hand_landmarks,
      mp_hands.HAND_CONNECTIONS,
      mp_drawing_styles.get_default_hand_landmarks_style(),
      mp_drawing_styles.get_default_hand_connections_style())

    # Get the top left corner of the detected hand's bounding box.
    height, width, _ = annotated_image.shape
    x_coordinates = [landmark.x for landmark in hand_landmarks]
    y_coordinates = [landmark.y for landmark in hand_landmarks]
    text_x = int(min(x_coordinates) * width)
    text_y = int(min(y_coordinates) * height) - MARGIN

    # Draw handedness (left or right hand) on the image.
    cv2.putText(annotated_image, f"{handedness[0].category_name}",
                (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

  return annotated_image

Будем считавать динамические жесты эмпирически, замеряя пройденный путь точек. Для измерения скорости точек лучше всего хранить их состояния в очереди.

We will evaluate dynamic gestures empirically by tracking the distance traveled by the landmarks. To calculate velocity, it is best to store their states in a queue.

In [ ]:
from collections import deque

index_tip_history = deque(maxlen=HISTORY_LENGTH)

In [ ]:
def update_history():
    global delta_x, delta_y
    
    if len(index_tip_history) == HISTORY_LENGTH:
        oldest_x, oldest_y = index_tip_history[0]
        cur_x, cur_y = index_tip_history[-1]
        
        delta_x = cur_x - oldest_x
        delta_y = cur_y - oldest_y
    else: 
        delta_x = 0
        delta_y = 0

Подключаемся к лампе по протоколу tuya
 
Connecting to the lightbulb using the Tuya protocol

In [ ]:
import tinytuya

d = tinytuya.BulbDevice(LAMP_ID, LAMP_IP, LAMP_KEY, version=3.5)
print(d.status())

brightness = d.get_brightness_percentage()
lamp_is_on =  brightness > 0

### Функции для манипулирования лампой
### Functions for lamp manipulation

In [ ]:
def async_lamp_command(command: str):
    global brightness
    global lamp_is_on
    global lamp_lock
    
    print(command)
    
    if command == "turn_off":
        d.turn_off(nowait=True)
    elif command == "turn_on": 
        d.turn_on(nowait=True)
    
    with lamp_lock:
        if command == "brightness_up":
            brightness += 30 * (abs(delta_x) + 1)
            if brightness > 100:
                brightness = 100
            print(brightness)
            d.set_brightness_percentage(brightness)
        elif command == "brightness_down":
            brightness -= 30 * (abs(delta_x) + 1)
            if brightness < 0:
                brightness = 0
            print(brightness)
            d.set_brightness_percentage(brightness)
            
        
    lamp_is_on = brightness > 0

In [ ]:
def trigger_action(action: str):
    global lamp_is_on    
    
    command = ''
    #with lamp_lock:
    if action == 'swipe_up':
        command = 'turn_on'
    elif action == 'swipe_down':
        command= 'turn_off'
    elif action == 'swipe_right':
        command = 'brightness_down'
    elif action == 'swipe_left':
        command = 'brightness_up'

    if not lamp_lock.locked() and command != '':
        t = threading.Thread(target=async_lamp_command, args=(command,),  daemon=True)
        t.start()
            
    index_tip_history.clear()

### Для того, чтобы понимать, какую точку считывать, используем таблицу
### We use a reference table to determine which specific point to track

![](media/mediapipe-hands.png)

### Основная функция-драйвер
### Main driver function

In [ ]:

with python.vision.HandLandmarker.create_from_options(options) as detector:
    screen = cv2.VideoCapture(0)
    screen.set(cv2.CAP_PROP_BUFFERSIZE, 2)
    
    while screen.isOpened():
        success, frame = screen.read()
        if not success:
            print("Не удалось получить кадр с камеры.")
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        
        timestamp_ms = int(time.time() * 1000)

        detector.detect_async(mp_image, timestamp_ms)
        
        annotated_frame = frame_rgb.copy() 
        
        if latest_result is not None:
            annotated_frame = draw_landmarks_on_image(annotated_frame, latest_result)
            
            hand_landmarks = latest_result.hand_landmarks[0]
            idx_tip = (hand_landmarks[8].x, hand_landmarks[8].y)
            idx_base = (hand_landmarks[0].x, hand_landmarks[0].y)
            
            distance = np.linalg.norm(np.array(idx_tip) - np.array(idx_base))
                        
            if distance < 0.3:
                index_tip_history.append(idx_tip)
            else:
                index_tip_history.clear()

            latest_result = None
        else: index_tip_history.clear()
            
        update_history()
        
        abs_dx = abs(delta_x)
        abs_dy = abs(delta_y)
            
        if max(abs_dx, abs_dy) > SWIPE_THRESHOLD:
            if abs_dx > abs_dy:
                if delta_x > SWIPE_THRESHOLD:
                    trigger_action("swipe_right")    
                elif delta_x < -SWIPE_THRESHOLD:
                    trigger_action("swipe_left")        
            else:
                if delta_y > SWIPE_THRESHOLD:
                    trigger_action("swipe_down")    
                elif delta_y < -SWIPE_THRESHOLD:
                    trigger_action("swipe_up")    

        final_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
        
        cv2.imshow('MediaPipe Hand Landmarker', final_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    screen.release()
    cv2.destroyAllWindows()